In [ ]:
import os
import re
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('../..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc
plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})

In [ ]:

ReVec = [
    20, 40, 50, 100, 150, 200
]

# folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_force_evaluation/archive/cylinder_bgk_u001"
# folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_force_evaluation/archive/cylinder_bgk_u003"
# folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_force_evaluation/archive/cylinder_bgk_u003_y20D"
# folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_force_evaluation/archive/cylinder_bgk_u003_y20D_spg0"
# folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_force_evaluation/cylinder_bgk_u003"
# folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_force_evaluation/cylinder_bgk_u003_y20D"
folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_force_evaluation/cylinder_bgk_u003_y20D_spg0"

folderVec = [
    f"{folder}/cylinder_dx25",
    f"{folder}/cylinder_dx32",
    f"{folder}/cylinder_dx40",
    f"{folder}/cylinder_dx50",
]

DLbVec = [
    25,
    32,
    40,
    50
]

markerVec = [
    'o',
    's',
    'd',
    '^',
    "P",
    '*',
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
    "#A17808",
]

# ULb = 0.01
ULb = 0.03

avgStartFactor = 0.8
stStartFactor = 0.3

labelVec = [
    "Present LBM BGK D/dx=25",
    "Present LBM BGK D/dx=32",
    "Present LBM BGK D/dx=40",
    "Present LBM BGK D/dx=50",
]

rhoPhy = 1.204
gamma = 1.0

dxVec = [1.0 / DLb for DLb in DLbVec]
ucVec = [uc(dx, rhoPhy, gamma) for dx in dxVec]

In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec

def extract_reynolds_number(path):
    match = re.search(r"Re(\d+)", path)
    if match:
        return int(match.group(1))
    return 0

def cal_coef_cylinder(force, area, rho, U):
    return force / (0.5 * rho * U**2 * area)

def cal_frontal_area_cylinder(diameter, span):
    return diameter * span


In [ ]:

ReVecVec = []
cdVecVec = []
clVecVec = []
stVecVec = []

stepVecVec = []
cxVecVec = []
cyVecVec = []
czVecVec = []

for case in range(0, len(folderVec)):
    pattern = f"{folderVec[case]}/cyl_dx{int(DLbVec[case])}_Re*/forcetorque/forceCylinder/forcetorque0000.txt"
    fileList = glob.glob(pattern)
    fileList = sorted(fileList, key=extract_reynolds_number)
    # print(fileList)
    ReVec = []
    avgCdVec = []
    avgClVec = []
    stVec = []

    stepVec = []
    cxVec = []
    cyVec = []
    czVec = []
    for ff in range(0, len(fileList)):
        Re = extract_reynolds_number(fileList[ff])
        forceX = read_col_probe(fileList[ff], 2, 2)
        forceY = read_col_probe(fileList[ff], 3, 2)
        forceZ = read_col_probe(fileList[ff], 4, 2)
        idxStart = int(avgStartFactor * len(forceX))
        area = cal_frontal_area_cylinder(DLbVec[case], 1)
        cd = cal_coef_cylinder(forceX, area, 1.0, ULb)
        cl = cal_coef_cylinder(forceY, area, 1.0, ULb)
        cz = cal_coef_cylinder(forceZ, area, 1.0, ULb)
        avgCd = np.mean(cd[idxStart:])
        avgCl = np.mean(cl[idxStart:])

        idxSt = int(stStartFactor * len(forceY))
        stepCol = read_col_probe(fileList[ff], 0, 2)
        dt = stepCol[1] - stepCol[0]
        freq = np.fft.rfftfreq(len(stepCol[idxSt:]), d=dt)
        fyFreqDomain = np.abs(np.fft.rfft(forceY[idxSt:]))
        stAll = freq * DLbVec[case] / ULb
        st = stAll[np.argmax(fyFreqDomain)]
        stVec.append(st)

        avgCdVec.append(avgCd)
        avgClVec.append(avgCl)
        ReVec.append(Re)

        stepVec.append(stepCol[idxSt:])
        cxVec.append(cd[idxSt:])
        cyVec.append(cl[idxSt:])
        czVec.append(cz[idxSt:])

    
    ReVecVec.append(ReVec)
    cdVecVec.append(avgCdVec)
    clVecVec.append(avgClVec)
    stVecVec.append(stVec)

    stepVecVec.append(stepVec)
    cxVecVec.append(cxVec)
    cyVecVec.append(cyVec)
    czVecVec.append(czVec)



In [ ]:

#! Drag coefficient
refCdFileVec = [
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/cd_Re_tritton_exp.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/cd_Re_takami_keller.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/cd_Re_fornberg.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/cd_Re_he.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/cd_Re_shu_lbm.csv"
]

refCdLabel = [
    'Tritton, Experiment',
    'Takami & Keller, Series expansion',
    'Fornberg, High order FDM',
    'He et al., LBM',
    'Shu et al., LBM'
]

refCdLine = [
    '',
    ':',
    '-.',
    '--',
    '-'
]

refCdMarker = [
    "x",
    "o",
    "D",
    "v",
    "s"
]

refReVec = []
refCdVec = []
for rr in range(0, len(refCdFileVec)):
    refReVec.append(read_csv_col(refCdFileVec[rr], 0, 1))
    refCdVec.append(read_csv_col(refCdFileVec[rr], 1, 1))

#! Strouhal number
refStFileVec = [
    # "/Users/maggie/repo/scripts/cases/force_evaluation/ref/st_Re_williamson.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/st_Re_hammache_gharib.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/st_Re_norberg.csv",
    "/Users/maggie/repo/scripts/cases/force_evaluation/ref/st_Re_shu_lbm.csv",

]

refStLabel = [
    # "Williamson",
    "Hammache & Gharib, Experiment",
    "Norberg, Experiment",
    "Shu et al., LBM",
]

refStMarker = [
    # ['o', 'none'],
    # ['o', 'full'],
    ['x', 'full'],
    ['^', 'none'],
    ['s', 'none']
]

refReStVec = []
refStVec = []
for rr in range(0, len(refStFileVec)):
    refReStVec.append(read_csv_col(refStFileVec[rr], 0, 1))
    refStVec.append(read_csv_col(refStFileVec[rr], 1, 1))

In [ ]:

#! Drag coefficient
figCd, axCd = plt.subplots(1, 1, figsize=(8, 6), facecolor='w', edgecolor='w')
# figCl, axCl = plt.subplots(1, 1, figsize=(8, 6), facecolor='w', edgecolor='w')

for rr in range(0, len(refCdFileVec)):
    axCd.plot(refReVec[rr], refCdVec[rr], lw=1, linestyle=refCdLine[rr], c='grey', marker=refCdMarker[rr], ms=8, fillstyle='none', label=refCdLabel[rr])

for case in range(0, len(folderVec)):
    axCd.plot(ReVecVec[case], cdVecVec[case], lw=0, marker=markerVec[case], c=colorVec[case], ms=8, mew=1.5, fillstyle='none', label=labelVec[case])
    # axCl.plot(ReVecVec[case], clVecVec[case], lw=0, marker=markerVec[case], c=colorVec[case], ms=8, mew=2, label=labelVec[case])

axCd.legend(loc='upper right', frameon=False)
# axCl.legend(loc='lower left', frameon=False)

axCd.set_xlim(0, 210)
axCd.set_ylim(0, 4)
axCd.set_xlabel("Re")
axCd.set_ylabel(r"$C_d$")
axCd.set_xticks(range(0, 221, 50))


#! Strouhal number
figSt, axSt = plt.subplots(1, 1, figsize=(8, 6), facecolor='w', edgecolor='w')
for rr in range(0, len(refStFileVec)):
    axSt.plot(refReStVec[rr], refStVec[rr], lw=0, c='grey', marker=refStMarker[rr][0], ms=8, fillstyle=refStMarker[rr][1], label=refStLabel[rr])

for case in range(0, len(folderVec)):
    axSt.plot(ReVecVec[case], stVecVec[case], lw=0, marker=markerVec[case], c=colorVec[case], ms=8, mew=1.5, fillstyle='none', label=labelVec[case])

axSt.set_xlim(0, 300)
axSt.set_ylim(0.1, 0.22)
axSt.set_xlabel("Re")
axSt.set_ylabel("St")
axSt.legend(loc='lower right', frameon=False)

In [ ]:
figC, axC = plt.subplots(3, 1, figsize=(10, 8), facecolor='w', edgecolor='w', sharex=True)

reynolds = 40
idxRe = int(np.abs(np.array(ReVec) - reynolds).argmin())

for case in range(0, len(folderVec)):
    axC[0].plot(stepVecVec[case][idxRe], cxVecVec[case][idxRe], label=labelVec[case], c=colorVec[case])
    axC[1].plot(stepVecVec[case][idxRe], cyVecVec[case][idxRe], label=labelVec[case], c=colorVec[case])
    axC[2].plot(stepVecVec[case][idxRe], czVecVec[case][idxRe], label=labelVec[case], c=colorVec[case])
    print(f"Case \"{labelVec[case]}\": Mean Cx={cdVecVec[case][idxRe]:.4f}, Mean Cy={clVecVec[case][idxRe]:.4f}, St={stVecVec[case][idxRe]:.4f}")
    print(f"               CyMin={np.min(cyVecVec[case][idxRe])}, CyMax={np.max(cyVecVec[case][idxRe])}")

axC[0].set_ylabel(r'$C_x$')
axC[1].set_ylabel(r'$C_y$')
axC[2].set_ylabel(r'$C_z$')
axC[2].set_xlabel('Step')
axC[0].legend(loc='upper right', frameon=False)